<a href="https://colab.research.google.com/github/LatikaaTejus/LatikaaTejus/blob/main/Toxic_msg_classify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.system('cp kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')

256

In [ ]:
!kaggle competitions download -c jigsaw-toxic-comment-classification-challenge

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


In [ ]:
import os
os.system('ls -la')

0

In [ ]:
!ls -la

total 67208
drwxr-xr-x 1 root root     4096 Jun 22 16:19 .
drwxr-xr-x 1 root root     4096 Jun 22 15:44 ..
drwxr-xr-x 4 root root     4096 Jun  4 13:32 .config
drwxr-xr-x 1 root root     4096 Jun  4 13:32 sample_data
-rw-r--r-- 1 root root 68802655 Jun 22 16:19 train.csv


In [17]:
import pandas as pd

df = pd.read_csv('/content/train.csv')
print(df.shape)
df.head()

(159571, 8)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [18]:
df.columns

Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='object')

In [19]:
df[df['toxic'] == 1]['comment_text'].sample(5, random_state=1).tolist()

["My page should be protected first so that worthless scum like you can't keep vandalizing it.",
 "Oh so it's you who wants the considerable amount of harm then is it? \n\nYou fucking cretin. You seriously think I'm going to stop vandalising just because you block me? I have a MASSIVE range of IPs and you will never block them all. I am never going to stop either. Hahaha 94.2.133.200",
 "I'll leave you alone when you levae me and my talk page alone, you white fuck.",
 "It's you who is doing original research by sucking dick of allmovie and not giving sources to why IB is an adventure film, Schindler's List was a man on a mission to save the jaws, is it an adventure film cocksucker? 201.68.139.99",
 "other people's\nGosh you can't even speak English which you claim to be your mother tongue. Puh-lease."]

In [20]:
df[df['toxic'] == 0]['comment_text'].sample(5, random_state=1).tolist()

['"\n\nInvitation and Recommendations for writing articles on Hindi Wikipedia\n\nHindi wikipedia invites and welcomes Wikipedians to contribute for the cause of spreading knowledge and the Hindi language. This page contains guidelines for writing a wiki-article on any topic at the Hindi Wikipedia, with special recommendations for writing in Hindi (Note: The script/font-family for Hindi is Devanāgari; the script/font-system for English is Roman script, also, the Hindi spelling system is not completely standardized). This article is yet in English language (mostly), in order to encourage even non-native/foreign people who have learnt/are learning Hindi to contribute to the Hindi wikipedia, and native Hindi speakers who normally write in English. The examples given below are only for explanation.\n\nRecommendations:\nFirstly for proper viewing, it is recommended to keep all links NOT-UNDERLINED. Otherwise the मात्रा below the Hindi alphabets might get partly hidden behind the underlines. 

In [21]:
df['toxic'].value_counts()
df['toxic'].value_counts(normalize=True)  # gives percentages

,proportion
toxic,
0,0.904156
1,0.095844


In [22]:
label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
df[label_cols].sum()

,0
toxic,15294
severe_toxic,1595
obscene,8449
threat,478
insult,7877
identity_hate,1405


In [23]:
df['comment_length'] = df['comment_text'].str.len()
df.groupby('toxic')['comment_length'].mean()

,comment_length
toxic,
0,404.549339
1,295.246044


In [24]:
toxic_sample = df[df['toxic'] == 1].sample(50, random_state=42)
nontoxic_sample = df[df['toxic'] == 0].sample(50, random_state=42)

sample_df = pd.concat([toxic_sample, nontoxic_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

sample_df['toxic'].value_counts()

,count
toxic,
0,50
1,50


In [26]:
!pip install -q google-genai

from google.colab import userdata
from google import genai

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

In [28]:
prompt_template = """Read the following comment and determine if it is toxic or not toxic.
Toxic means rude, disrespectful, insulting, threatening, or hateful in a way that could make someone want to leave a conversation.

Comment: "{comment}"

Respond with only one word: either "toxic" or "not_toxic". Do not explain your answer."""

In [32]:
from google.colab import userdata
from huggingface_hub import InferenceClient

hf_token = userdata.get('HF_TOKEN')
client_hf = InferenceClient(token=hf_token)

In [34]:
from huggingface_hub import InferenceClient

client_hf = InferenceClient(token=hf_token)

response = client_hf.chat_completion(
    model="moonshotai/Kimi-K2-Instruct-0905",
    messages=[{"role": "user", "content": "Say hello in one word."}],
    max_tokens=20
)
print(response.choices[0].message.content)

Hello


In [35]:
test_comment = sample_df.iloc[0]['comment_text']
prompt = prompt_template.format(comment=test_comment)

response = client_hf.chat_completion(
    model="moonshotai/Kimi-K2-Instruct-0905",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=20
)

print("Comment:", test_comment)
print("Actual label:", sample_df.iloc[0]['toxic'])
print("Model said:", response.choices[0].message.content)

Comment: I'm so glad I have your approval.  07:25, 24 Feb 2004 (UTC)
Actual label: 0
Model said: not_toxic


In [36]:
import time

predictions = []

for i, row in sample_df.iterrows():
    prompt = prompt_template.format(comment=row['comment_text'])
    try:
        response = client_hf.chat_completion(
            model="moonshotai/Kimi-K2-Instruct-0905",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=20
        )
        result = response.choices[0].message.content.strip().lower()
    except Exception as e:
        result = "error"
        print(f"Row {i} failed: {e}")

    predictions.append(result)
    time.sleep(1)  # small pause to avoid rate limits

sample_df['prediction_zero_shot'] = predictions
print(sample_df['prediction_zero_shot'].value_counts())

Row 44 failed: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a3968e0-306568d11dc7c4ca7986be1e;7d854e99-c6e0-4a96-b8fd-6449807c8313)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
Row 45 failed: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a3968e1-01d134383e9ef8f940718e3d;a55125d2-d50a-41b0-9932-736bbbbaa00a)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
Row 46 failed: Client error '402 Payment Required' for url

In [37]:
# Filter out errors first
clean_df = sample_df[sample_df['prediction_zero_shot'] != 'error'].copy()

# Convert model's text answer into 0/1 to match the real labels
clean_df['predicted_label'] = clean_df['prediction_zero_shot'].apply(lambda x: 1 if x == 'toxic' else 0)

# Compare predicted vs actual
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

acc = accuracy_score(clean_df['toxic'], clean_df['predicted_label'])
prec = precision_score(clean_df['toxic'], clean_df['predicted_label'])
rec = recall_score(clean_df['toxic'], clean_df['predicted_label'])

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print(confusion_matrix(clean_df['toxic'], clean_df['predicted_label']))

Accuracy: 0.925
Precision: 0.8863636363636364
Recall: 0.975
[[35  5]
 [ 1 39]]
